# Анализ данных криптовалютного рынка для обнаружения волатильных паттернов и корреляций

**Щеглов Александр · ИД24-2**

## 1. Объект исследования и задачи начального анализа

**Объект исследования** — криптовалютный рынок, представленный выбранными монетами
и периодами наблюдения в датасетах.

Для описания его состояния используются цена, доходность, объём торгов и готовые
показатели волатильности. В качестве предполагаемых факторов влияния рассматриваются
настроения участников рынка, макроэкономические показатели и информационный фон.
Их фактическое влияние на этом этапе ещё не проверяется.

Основная единица наблюдения в рыночном CSV — **одна криптовалюта за один день**.
В остальных файлах это четырёхчасовая свеча BTC, новостная публикация или запись о торговом инструменте.

In [1]:
from pathlib import Path
import json
import xml.etree.ElementTree as ET
import pandas as pd

try:
    from IPython.display import display
except ImportError:
    def display(obj):
        print(obj.to_string() if isinstance(obj, (pd.DataFrame, pd.Series)) else obj)

DATA_DIR = Path(".")  # папка с четырьмя исходными файлами


## 2. Загрузка и назначение датасетов

### 2.1. Табличные рыночные данные — CSV

`crypto_market_cycles_2017_present.csv` содержит цены, объёмы, готовые показатели
доходности и волатильности, сведения о рыночных циклах и внешнем контексте.
Это основной источник для описания состояния рынка.

Для анализа читается **весь файл**. Первые пять строк используются только для просмотра.
CSV представляет табличные данные; сам файл не является реляционной базой данных.


In [2]:
path = DATA_DIR / "crypto_market_cycles_2017_present.csv"
df_market = pd.read_csv(path) if path.exists() else None
print("Рыночный CSV загружен." if df_market is not None else f"Нет файла: {path.name}")


Рыночный CSV загружен.


### 2.2. Иерархические данные — XML

`crypto-tradeing bots.xml` описывает торговые боты, стратегии и поддерживаемые блокчейны.
Записи внутри разделов преобразуются в таблицу для проверки заполненности.
Тип раздела сохраняется в `record_type`, идентификатор записи — в `record_id`.

На начальном этапе XML служит справочником инструментов. Без журнала сделок нельзя
установить, как активность конкретного бота связана с движением цены.


In [3]:
path = DATA_DIR / "crypto-tradeing bots.xml"
df_bots = None
if path.exists():
    root = ET.parse(path).getroot()
    rows = []
    for section in root.findall("section"):
        for record in section.findall("record"):
            row = {child.tag: " ".join(child.itertext()).strip() for child in record}
            row["record_id"] = record.get("id")
            row["record_type"] = row.get("record_type") or section.get("record_type")
            rows.append(row)
    df_bots = pd.DataFrame(rows)
    print("Корневой тег:", root.tag, "| Записей:", len(df_bots))
else:
    print(f"Нет файла: {path.name}")


Корневой тег: dataset | Записей: 220


### 2.3. Четырёхчасовые свечи BTC — JSON

`btcusdt_4h_fear_greed.json` содержит цены BTC и индекс страха и жадности.
JSON — полуструктурированный формат. Его записи преобразуются в таблицу.
В дальнейшем этот источник позволит рассмотреть рынок подробнее, чем дневной CSV.
Сейчас источники изучаются отдельно.


In [4]:
path = DATA_DIR / "btcusdt_4h_fear_greed.json"
df_json = None
if path.exists():
    with path.open(encoding="utf-8-sig") as f:
        records = json.load(f)
    if isinstance(records, dict):
        records = records.get("data", records.get("records", [records]))
    df_json = pd.DataFrame(records)
    print("JSON загружен. Записей:", len(df_json))
else:
    print(f"Нет файла: {path.name}")


JSON загружен. Записей: 14280


### 2.4. Новости — CSV с неструктурированным текстом

`crypto_news.csv` содержит заголовки, описания, источники и даты публикаций.
CSV задаёт табличную структуру, а поля `title` и `description` содержат
неструктурированный текст. Они представляют информационный фон рынка.
На этом этапе изучаются полнота текстов и состав источников.


In [5]:
path = DATA_DIR / "crypto_news.csv"
df_news = pd.read_csv(path) if path.exists() else None
print("Новостной CSV загружен." if df_news is not None else f"Нет файла: {path.name}")

datasets = {
    "Рынок (CSV)": df_market,
    "Боты (XML)": df_bots,
    "Свечи BTC (JSON)": df_json,
    "Новости (CSV)": df_news,
}
# Пустые строки и строки из пробелов тоже считаем пропусками.
for frame in datasets.values():
    if frame is not None:
        frame.replace(r"^\s*$", pd.NA, regex=True, inplace=True)


Новостной CSV загружен.


## 3. Первичное знакомство со структурой

Для каждого доступного файла посмотрим число строк и столбцов, названия полей,
первые записи и количество полных дубликатов.
Повторные записи пока только подсчитываются: перед удалением нужно понять, почему они появились.


In [6]:
overview_rows = []
for name, frame in datasets.items():
    overview_rows.append({
        "Датасет": name,
        "Файл подключён": frame is not None,
        "Строк": len(frame) if frame is not None else pd.NA,
        "Столбцов": len(frame.columns) if frame is not None else pd.NA,
    })
    if frame is not None:
        print(f"\n{name}")
        print("Столбцы:", list(frame.columns))
        print("Полных дубликатов сверх первой записи:", frame.astype("string").duplicated().sum())
        display(frame.head())
display(pd.DataFrame(overview_rows))



Рынок (CSV)
Столбцы: ['date', 'coin', 'open', 'high', 'low', 'close', 'volume', 'daily_return_pct', 'log_return', 'rolling_7d_vol', 'rolling_30d_vol', 'ath_usd', 'drawdown_from_ath_pct', 'sma_50d', 'sma_200d', 'golden_cross', 'fear_greed_value', 'fear_greed_label', 'fear_greed_zone', 'btc_dominance_pct', 'eth_dominance_pct', 'total_market_cap_usd', 'btc_market_cap_usd', 'eth_market_cap_usd', 'halving_epoch', 'days_since_halving', 'days_to_next_halving', 'is_halving_day', 'cycle_progress_pct', 'block_reward_btc', 'dxy_close', 'us10y_yield']
Полных дубликатов сверх первой записи: 0


,date,coin,open,high,low,close,volume,daily_return_pct,log_return,rolling_7d_vol,...,btc_market_cap_usd,eth_market_cap_usd,halving_epoch,days_since_halving,days_to_next_halving,is_halving_day,cycle_progress_pct,block_reward_btc,dxy_close,us10y_yield
0,2017-11-09,ADA,0.025160,0.035060,0.025006,0.032053,18716200,NaN,NaN,NaN,...,1.698874e+12,3.281062e+11,2,488,914,0,33.47,12.5,94.389999,2.400
1,2017-11-10,ADA,0.032219,0.033348,0.026451,0.027119,6766780,-15.393260,-0.167156,NaN,...,1.698874e+12,3.281062e+11,2,489,913,0,33.54,12.5,94.389999,2.400
2,2017-11-11,ADA,0.026891,0.029659,0.025684,0.027437,5532220,1.172610,0.011658,11.713839,...,1.698874e+12,3.281062e+11,2,490,912,0,33.61,12.5,94.389999,2.400
3,2017-11-12,ADA,0.027480,0.027952,0.022591,0.023977,7280250,-12.610706,-0.134797,8.870836,...,1.698874e+12,3.281062e+11,2,491,911,0,33.68,12.5,93.660004,2.354
4,2017-11-13,ADA,0.024364,0.026300,0.023495,0.025808,4419440,7.636480,0.073589,11.008518,...,1.698874e+12,3.281062e+11,2,492,910,0,33.74,12.5,93.660004,2.354



Боты (XML)
Столбцы: ['record_type', 'question', 'answer', 'bot_category', 'chain', 'section', 'model', 'provider', 'source_url', 'bot_url', 'tutorial_url', 'reliability', 'consensus', 'critical_note', 'record_id']
Полных дубликатов сверх первой записи: 0


,record_type,question,answer,bot_category,chain,section,model,provider,source_url,bot_url,tutorial_url,reliability,consensus,critical_note,record_id
0,bot_profile,"What is ArbitrageScanner, which chains does it...",Category: Arbitrage Scanner / CEX + DEX | Chai...,Arbitrage Scanner / CEX + DEX,40+ blockchains,bot_profile,<NA>,<NA>,https://telegramtrading.net/best-crypto-arbitr...,https://telegramtrading.net/ArbitrageScanner,<NA>,VERIFIED,False,False,profile_arbitragescanner
1,bot_profile,"What is Axiom, which chains does it support, a...",Category: DEX Terminal / Solana | Chains (2): ...,DEX Terminal / Solana,Solana | Hyperliquid (L1 perpetuals),bot_profile,<NA>,<NA>,https://telegramtrading.net/,<NA>,<NA>,VERIFIED,False,False,profile_axiom
2,bot_profile,"What is Banana Gun, which chains does it suppo...",Category: DEX Sniper / Multi-chain | Chain foc...,DEX Sniper / Multi-chain,Ethereum | Base | BNB Chain | MegaETH | Robinh...,bot_profile,<NA>,<NA>,https://telegramtrading.net/banana-gun-bot-rev...,https://telegramtrading.net/bananagun,<NA>,VERIFIED,False,False,profile_banana_gun
3,bot_profile,"What is Bloom, which chains does it support, a...",Category: DEX Sniper / Multi-chain | Chain foc...,DEX Sniper / Multi-chain,Solana | Ethereum | BNB Chain | Base | Hyperli...,bot_profile,<NA>,<NA>,https://telegramtrading.net/bloom-bot-review/,<NA>,<NA>,VERIFIED,False,False,profile_bloom
4,bot_profile,"What is BonkBot + Telemetry, which chains does...",Category: DEX Sniper / Solana | Chains (1): So...,DEX Sniper / Solana,Solana,bot_profile,<NA>,<NA>,https://telegramtrading.net/bonkbot-telemetry-...,https://telegramtrading.net/bonkbot,<NA>,VERIFIED,False,False,profile_bonkbot



Свечи BTC (JSON)
Столбцы: ['timestamp', 'open', 'close', 'high', 'low', 'Fear & Greed Index', 'Fear & Greed Classification']
Полных дубликатов сверх первой записи: 0


,timestamp,open,close,high,low,Fear & Greed Index,Fear & Greed Classification
0,2026-09-30 04:00:00,83640.0,83269.0,83685.0,83164.0,71.0,Greed
1,2026-09-30 00:00:00,83566.0,83640.0,83811.0,83368.0,71.0,Greed
2,2026-09-29 20:00:00,83080.0,83566.0,83663.0,82911.0,73.0,Greed
3,2026-09-29 16:00:00,84331.0,83080.0,84486.0,82936.0,73.0,Greed
4,2026-09-29 12:00:00,83978.0,84339.0,84442.0,83747.0,73.0,Greed



Новости (CSV)
Столбцы: ['url', 'title', 'description', 'thumbnail', 'createdAt', 'source']
Полных дубликатов сверх первой записи: 0


,url,title,description,thumbnail,createdAt,source
0,https://www.coindesk.com/markets/2026/08/15/ro...,Robot maker Unitree is going public. Hyperliqu...,Hyperliquid traders value Unitree at nearly $3...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 16:00:00 +0000",coindesk
1,https://www.coindesk.com/business/2026/08/15/s...,Swiss mega-bank UBS ramps up its Bitcoin expos...,"Direct holdings of IBIT also rose 12% to 407,8...",https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:56:05 +0000",coindesk
2,https://www.coindesk.com/policy/2026/08/15/why...,Why the world’s second-largest Bitcoin mining ...,The Energy Ministry enacted the year-round res...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:21:15 +0000",coindesk
3,https://www.coindesk.com/business/2026/08/15/p...,Paul Tudor Jones’ investment firm increases st...,"Calls fell 85.2% to 148,000 underlying shares,...",https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 15:12:29 +0000",coindesk
4,https://www.coindesk.com/business/2026/08/15/t...,The $11.2 billion in 2026 funding that killed ...,Dubai-based crypto lawyer Irina Heaver and her...,https://cdn.sanity.io/images/s3y3vcno/producti...,"Sat, 15 Aug 2026 14:00:00 +0000",coindesk


,Датасет,Файл подключён,Строк,Столбцов
0,Рынок (CSV),True,30120,32
1,Боты (XML),True,220,15
2,Свечи BTC (JSON),True,14280,7
3,Новости (CSV),True,413,6


## 4. Выбор числовых и категориальных характеристик

**Числовые характеристики** — измеряемые величины: цена, доходность, объём, значение индекса.
**Категориальные характеристики** — группы и метки: монета, источник новости, тип стратегии.

Тип хранения и смысл признака могут различаться. Например, `halving_epoch` хранится
числом, но обозначает категорию цикла; `golden_cross` и `is_halving_day` — бинарные признаки.
Даты относятся к временным характеристикам, заголовки и описания — к текстовым.
Идентификаторы и ссылки помогают различать записи, но сами по себе не описывают состояние рынка.

### 4.1. Характеристики основного рыночного CSV

| Признак | Тип по смыслу | Роль в аналитическом портрете |
|---|---|---|
| `close` | Числовой | Уровень цены закрытия |
| `daily_return_pct` | Числовой | Направление и величина дневного изменения цены |
| `rolling_7d_vol`, `rolling_30d_vol` | Числовые | Готовые оценки изменчивости на разных окнах |
| `drawdown_from_ath_pct` | Числовой | Снижение относительно максимума, указанного источником |
| `volume` | Числовой | Торговая активность; предполагаемый фактор связи с колебаниями |
| `fear_greed_value` | Числовой | Настроения участников рынка |
| `btc_dominance_pct` | Числовой | Доля Bitcoin и структура рынка |
| `total_market_cap_usd` | Числовой | Общий масштаб рынка |
| `dxy_close`, `us10y_yield` | Числовые | Внешний макроэкономический контекст |
| `days_since_halving` | Числовой | Положение наблюдения относительно халвинга Bitcoin |
| `coin` | Категориальный | Какая криптовалюта наблюдается |
| `fear_greed_label` | Категориальный | Группа рыночных настроений |
| `halving_epoch` | Категориальный | Эпоха халвинга Bitcoin |
| `golden_cross` | Бинарный категориальный | Технический сигнал или состояние тренда — уточнить по описанию источника |
| `is_halving_day` | Бинарный категориальный | Признак дня халвинга |

Таким образом, **цену, доходность, волатильность и просадку** выбираем для описания
состояния объекта. **Объём, настроения, рыночные и макроэкономические показатели**
рассматриваем как кандидатов для дальнейшего исследования связей.
Выбор признака ещё не доказывает его влияние.

Для первого этапа достаточно этого набора. `open`, `high`, `low` и остальные поля
сохраняются в исходной таблице и могут быть использованы позднее.
Перед сравнением готовых показателей нужно уточнить единицы объёма и методику расчёта волатильности.


### 4.2. Характеристики остальных источников

| Датасет | Числовые | Категориальные | Временные / текстовые и назначение |
|---|---|---|---|
| JSON BTC | `open`, `high`, `low`, `close`, `Fear & Greed Index` | `Fear & Greed Classification` | `timestamp` — время свечи; цены описывают состояние BTC, индекс — настроения |
| XML ботов | Готовые рыночные числовые признаки не выбираются | `record_type`, `bot_category`, `chain` | `record_id` — идентификатор; `question`, `answer` — текстовый контекст |
| Новости | На этом этапе числовые признаки не создаются | `source` | `createdAt` — дата; `title`, `description` — информационный фон; `url` — ссылка |

Не каждый датасет должен содержать оба типа признаков. XML полезен как справочник,
а новости — как текстовый источник. Их не нужно искусственно превращать в набор чисел
на первом этапе работы.


In [7]:
# Выбранные признаки. Берём только те, которые действительно есть в файлах.
numeric_features = {
    "Рынок (CSV)": ["close", "daily_return_pct", "rolling_7d_vol", "rolling_30d_vol",
                    "drawdown_from_ath_pct", "volume", "fear_greed_value",
                    "btc_dominance_pct", "total_market_cap_usd", "dxy_close",
                    "us10y_yield", "days_since_halving"],
    "Свечи BTC (JSON)": ["open", "high", "low", "close", "Fear & Greed Index"],
}
categorical_features = {
    "Рынок (CSV)": ["coin", "fear_greed_label", "halving_epoch", "golden_cross", "is_halving_day"],
    "Свечи BTC (JSON)": ["Fear & Greed Classification"],
    "Боты (XML)": ["record_type", "bot_category", "chain"],
    "Новости (CSV)": ["source"],
}
date_columns = {"Рынок (CSV)": "date", "Свечи BTC (JSON)": "timestamp", "Новости (CSV)": "createdAt"}

for name, frame in datasets.items():
    if frame is not None:
        print(f"\n{name}")
        print("Выбраны числовые:", [c for c in numeric_features.get(name, []) if c in frame])
        print("Выбраны категориальные:", [c for c in categorical_features.get(name, []) if c in frame])



Рынок (CSV)
Выбраны числовые: ['close', 'daily_return_pct', 'rolling_7d_vol', 'rolling_30d_vol', 'drawdown_from_ath_pct', 'volume', 'fear_greed_value', 'btc_dominance_pct', 'total_market_cap_usd', 'dxy_close', 'us10y_yield', 'days_since_halving']
Выбраны категориальные: ['coin', 'fear_greed_label', 'halving_epoch', 'golden_cross', 'is_halving_day']

Боты (XML)
Выбраны числовые: []
Выбраны категориальные: ['record_type', 'bot_category', 'chain']

Свечи BTC (JSON)
Выбраны числовые: ['open', 'high', 'low', 'close', 'Fear & Greed Index']
Выбраны категориальные: ['Fear & Greed Classification']

Новости (CSV)
Выбраны числовые: []
Выбраны категориальные: ['source']


## 5. Проверка типов и исследование пропусков

Числовые значения иногда записаны строками. Выбранные числовые поля приведём
к числовому типу, даты — к временному. Нераспознанные значения станут пропусками;
их количество отдельно покажет таблица преобразований.
Время приводится к UTC; для дат без часового пояса это рабочее допущение,
которое нужно подтвердить по описанию датасета.

**Исходные файлы не изменяются.** Преобразования выполняются в загруженных таблицах.


In [8]:
conversion_rows = []
for name, frame in datasets.items():
    if frame is None:
        continue
    columns = [c for c in numeric_features.get(name, []) if c in frame]
    date_col = date_columns.get(name)
    if date_col in frame:
        columns.append(date_col)
    for col in columns:
        missing_before = frame[col].isna().sum()
        if col == date_col:
            frame[col] = pd.to_datetime(frame[col], errors="coerce", format="mixed", utc=True)
        else:
            frame[col] = pd.to_numeric(frame[col], errors="coerce")
        conversion_rows.append({"Датасет": name, "Поле": col,
                                "Новых пропусков после преобразования": frame[col].isna().sum() - missing_before})

if conversion_rows:
    display(pd.DataFrame(conversion_rows))
    for name, col in date_columns.items():
        frame = datasets[name]
        if frame is not None and col in frame:
            print(f"{name}: {frame[col].min()} → {frame[col].max()}")
else:
    print("Проверка типов ожидает исходные файлы.")


,Датасет,Поле,Новых пропусков после преобразования
0,Рынок (CSV),close,0
1,Рынок (CSV),daily_return_pct,0
2,Рынок (CSV),rolling_7d_vol,0
3,Рынок (CSV),rolling_30d_vol,0
4,Рынок (CSV),drawdown_from_ath_pct,0
5,Рынок (CSV),volume,0
6,Рынок (CSV),fear_greed_value,0
7,Рынок (CSV),btc_dominance_pct,0
8,Рынок (CSV),total_market_cap_usd,0
9,Рынок (CSV),dxy_close,0


Рынок (CSV): 2017-01-01 00:00:00+00:00 → 2026-09-26 00:00:00+00:00
Свечи BTC (JSON): 2020-03-25 08:00:00+00:00 → 2026-09-30 04:00:00+00:00
Новости (CSV): 2026-07-01 17:51:50+00:00 → 2026-08-15 17:26:43+00:00


### 5.1. Количество и доля пропусков по столбцам

Для каждого поля рассчитывается:

$$\text{Доля пропусков, %} = \frac{\text{Число пропущенных значений}}{\text{Число строк}}\cdot 100.$$

Также показываются тип хранения и число уникальных заполненных значений.
Это помогает заметить полностью пустые поля и признаки, имеющие только одно значение.
Количество строк с хотя бы одним пропуском характеризует заполненность всей таблицы,
но само по себе не означает, что эти строки нужно удалять.


In [9]:
missing_reports = {}
for name, frame in datasets.items():
    if frame is None:
        continue
    report = pd.DataFrame({
        "Тип хранения": frame.dtypes.astype(str),
        "Пропусков": frame.isna().sum(),
        "Пропусков, %": (frame.isna().mean() * 100).round(2),
        "Уникальных без пропусков": frame.astype("string").nunique(),
    }).sort_values("Пропусков, %", ascending=False)
    missing_reports[name] = report
    print(f"\n{name}: {len(frame)} строк")
    display(report)
    print("Строк с хотя бы одним пропуском:", frame.isna().any(axis=1).sum())
    print("Полностью пустых столбцов:", list(frame.columns[frame.isna().all()]))

if not missing_reports:
    print("Для расчёта пропусков добавьте исходные CSV, JSON и XML.")



Рынок (CSV): 30120 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
fear_greed_label,object,2342,7.78,5
fear_greed_zone,object,2342,7.78,5
fear_greed_value,float64,2342,7.78,90
rolling_30d_vol,float64,20,0.07,30033
rolling_7d_vol,float64,20,0.07,30035
daily_return_pct,float64,10,0.03,30057
log_return,float64,10,0.03,26895
high,float64,0,0.00,29838
volume,int64,0,0.00,30118
close,float64,0,0.00,29856


Строк с хотя бы одним пропуском: 2348
Полностью пустых столбцов: []

Боты (XML): 220 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
bot_url,object,202,91.82,18
provider,object,176,80.00,6
model,object,176,80.00,6
tutorial_url,object,152,69.09,33
chain,object,129,58.64,36
bot_category,object,82,37.27,33
section,object,69,31.36,62
question,object,44,20.00,176
reliability,object,44,20.00,3
record_type,object,0,0.00,6


Строк с хотя бы одним пропуском: 220
Полностью пустых столбцов: []

Свечи BTC (JSON): 14280 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
Fear & Greed Classification,object,6,0.04,5
Fear & Greed Index,float64,6,0.04,88
timestamp,"datetime64[ns, UTC]",0,0.00,14280
close,float64,0,0.00,13786
open,float64,0,0.00,13791
low,float64,0,0.00,13478
high,float64,0,0.00,13509


Строк с хотя бы одним пропуском: 6
Полностью пустых столбцов: []

Новости (CSV): 413 строк


,Тип хранения,Пропусков,"Пропусков, %",Уникальных без пропусков
thumbnail,object,1,0.24,403
url,object,0,0.00,413
title,object,0,0.00,412
description,object,0,0.00,412
createdAt,"datetime64[ns, UTC]",0,0.00,412
source,object,0,0.00,10


Строк с хотя бы одним пропуском: 1
Полностью пустых столбцов: []


### 5.2. Простое уточнение: пропуски внутри групп

В рыночном CSV полезно проверить, одинаково ли заполнены данные разных монет.
В XML поля могут относиться только к определённым типам записей: например,
отсутствие `bot_category` у вопроса и ответа не обязательно является ошибкой.
Поэтому дополнительно посмотрим долю пропусков внутри этих групп.


In [10]:
if df_market is not None and "coin" in df_market:
    cols = [c for c in ["close", "volume", "fear_greed_value", "rolling_30d_vol"] if c in df_market]
    if cols:
        print("Рыночный CSV: пропуски по монетам, %")
        display(df_market.groupby("coin", dropna=False)[cols].agg(lambda s: s.isna().mean() * 100).round(2))

if df_bots is not None and "record_type" in df_bots:
    cols = [c for c in ["bot_category", "chain", "question", "answer"] if c in df_bots]
    if cols:
        print("XML: пропуски по типам записей, %")
        display(df_bots.groupby("record_type", dropna=False)[cols].agg(lambda s: s.isna().mean() * 100).round(2))


Рыночный CSV: пропуски по монетам, %


,close,volume,fear_greed_value,rolling_30d_vol
coin,,,,
ADA,0.0,0.0,7.55,0.06
AVAX,0.0,0.0,0.05,0.09
BNB,0.0,0.0,7.55,0.06
BTC,0.0,0.0,15.66,0.06
DOGE,0.0,0.0,7.55,0.06
DOT,0.0,0.0,0.04,0.09
ETH,0.0,0.0,7.55,0.06
LTC,0.0,0.0,15.66,0.06
SOL,0.0,0.0,0.04,0.08


XML: пропуски по типам записей, %


,bot_category,chain,question,answer
record_type,,,,
bot_profile,0.0,11.11,0.0,0.0
chain_profile,0.0,0.00,0.0,0.0
llm_consensus,0.0,100.00,100.0,0.0
qa_pair,0.0,0.00,0.0,0.0
real_test_data,100.0,100.00,0.0,0.0
verified_fact,100.0,100.00,0.0,0.0


### 5.3. Как интерпретировать пропуски на начальном этапе

| Где обнаружен пропуск | Возможная причина — требует проверки | Первоначальное решение |
|---|---|---|
| Первая доходность | Нет предыдущей цены | Сохранить пропуск |
| Начало `rolling_7d_vol`, `rolling_30d_vol` | Недостаточно дней для окна | Сохранить пропуск; проверить правило расчёта |
| `fear_greed_value`, `dxy_close`, `us10y_yield` | Другой период покрытия или календарь источника | Уточнить доступность данных по датам |
| `close`, `date`, `coin` | Неполная запись или ошибка чтения | Проверить отдельно: это важные поля наблюдения |
| `description` новости | Источник не передал описание | Использовать доступный заголовок при дальнейшем анализе |
| Поле XML только у некоторых типов записей | Поле неприменимо к другим типам | Оценивать полноту внутри `record_type` |

Сейчас достаточно зафиксировать пропуски и возможные причины.
**Автоматическое заполнение нулями, средними значениями или массовое удаление строк
на этом этапе не выполняется.** Сначала нужно понять смысл каждого поля.


## 6. Минимальная описательная статистика выбранных признаков

Для числовых полей посмотрим число заполненных значений, среднее, медиану,
минимум и максимум. Эти значения помогают заметить необычный масштаб или возможные ошибки.
Например, у цены нужно проверить нулевые и отрицательные значения, а у индекса
страха и жадности — значения вне шкалы 0–100.

Для категориальных полей посчитаем частоты значений. Пропуск будет показан
отдельной строкой. Максимум десять наиболее частых категорий выводятся для краткости.


In [11]:
for name, frame in datasets.items():
    if frame is None or frame.empty:
        continue
    cols = [c for c in numeric_features.get(name, []) if c in frame]
    if cols:
        print(f"\n{name}: числовые характеристики")
        stats = frame[cols].describe().T[["count", "mean", "50%", "min", "max"]]
        stats.columns = ["Заполнено", "Среднее", "Медиана", "Минимум", "Максимум"]
        display(stats)
    for col in categorical_features.get(name, []):
        if col in frame:
            print(f"{name}: {col}")
            display(frame[col].value_counts(dropna=False).head(10).rename("Количество").to_frame())



Рынок (CSV): числовые характеристики


,Заполнено,Среднее,Медиана,Минимум,Максимум
close,30120.0,4.556112e+03,2.738360e+01,1.038000e-03,1.247525e+05
daily_return_pct,30110.0,2.412812e-01,9.127000e-03,-4.234722e+01,3.555466e+02
rolling_7d_vol,30100.0,4.316715e+00,3.522285e+00,1.304300e-01,1.372630e+02
rolling_30d_vol,30100.0,4.709001e+00,4.034044e+00,1.304300e-01,6.753813e+01
drawdown_from_ath_pct,30120.0,-6.227310e+01,-7.061443e+01,-9.860659e+01,0.000000e+00
volume,30120.0,6.030088e+09,1.139117e+09,3.957000e+03,3.509679e+11
fear_greed_value,27778.0,4.667370e+01,4.700000e+01,5.000000e+00,9.500000e+01
btc_dominance_pct,30120.0,5.202889e+01,5.416000e+01,3.500000e+01,8.700000e+01
total_market_cap_usd,30120.0,2.888259e+12,2.888259e+12,2.888259e+12,2.888259e+12
dxy_close,30120.0,9.917168e+01,9.881000e+01,8.907000e+01,1.133100e+02


Рынок (CSV): coin


,Количество
coin,
LTC,3556
BTC,3556
XRP,3244
ADA,3244
DOGE,3244
BNB,3244
ETH,3244
SOL,2361
DOT,2229


Рынок (CSV): fear_greed_label


...

,Заполнено,Среднее,Медиана,Минимум,Максимум
open,14280.0,51488.636947,46336.25,5873.0,125496.0
high,14280.0,51875.182892,46761.35,5950.0,126080.0
low,14280.0,51085.205014,45710.00,5841.5,124863.0
close,14280.0,51494.136597,46343.85,5873.0,125375.0
Fear & Greed Index,14274.0,47.950960,49.00,5.0,95.0


Свечи BTC (JSON): Fear & Greed Classification


,Количество
Fear & Greed Classification,
Greed,4118
Fear,3444
Extreme Fear,3154
Neutral,1992
Extreme Greed,1566
None,6


Новости (CSV): source


,Количество
source,
cryptodaily,200
bsc,40
cryptopotato,36
decrypt,35
cointelegraph,30
coindesk,25
theguardian,20
bitcoinmagazine,10
coinjournal,9


# 2.3. Подготовка данных: предварительный анализ, очистка, слияние и хранение


**Рабочая гипотеза для выбора признаков:** изменчивость криптовалютного рынка может
быть связана с торговой активностью, настроениями, макроэкономическим контекстом и
информационным фоном. Здесь мы готовим данные для её последующей проверки,
а не делаем вывод о наличии или причинности этих связей.

**План:** описать атрибуты и шкалы → очистить записи → аннотировать и векторизовать
тексты → привести источники к совместимому уровню → объединить → сохранить и обновлять.


## 2.3.1. Шкалы, физический смысл и аналитические роли атрибутов

Используем номинальную шкалу для названий, порядковую для упорядоченных групп,
интервальную для времени и приращений, шкалу отношений для положительных измеряемых
величин с осмысленным нулём. Свободный текст сначала не относится к числовой шкале.

Аналитические роли: ключ, время, характеристика состояния/риска, предполагаемый фактор,
группировочный признак, справочный или служебный атрибут.
У Fear & Greed равенство интервалов между баллами не считается доказанным.
`halving_epoch` — категория эпохи, а числовые `golden_cross` и `is_halving_day` — флаги.

Ниже подготовлен словарь всех полей, известных из исходного ноутбука.
Поля, которые обнаружатся дополнительно, будут помечены как требующие описания.
Неизвестные единицы и методики явно указаны: они не восстанавливаются по одному имени файла.
Полный словарь, включая получаемые атрибуты, также находится в `attribute_dictionary.csv` в архиве.


In [12]:
from pathlib import Path
import argparse
import hashlib
import json
import re
import tempfile
import uuid
from datetime import datetime, timezone
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd
from scipy import sparse
from sklearn.feature_extraction.text import TfidfVectorizer

FILES = {"market": "crypto_market_cycles_2017_present.csv",
         "intraday": "btcusdt_4h_fear_greed.json",
         "news": "crypto_news.csv", "bots": "crypto-tradeing bots.xml"}
KEYS = {"market": ["coin", "date"], "intraday": ["timestamp"],
        "news": ["news_id"], "bots": ["record_id"]}
MARKET_NUMERIC = ["open", "high", "low", "close", "volume", "daily_return_pct", "log_return",
                  "rolling_7d_vol", "rolling_30d_vol", "ath_usd", "drawdown_from_ath_pct",
                  "sma_50d", "sma_200d", "fear_greed_value", "btc_dominance_pct",
                  "eth_dominance_pct", "total_market_cap_usd", "btc_market_cap_usd",
                  "eth_market_cap_usd", "halving_epoch", "days_since_halving",
                  "days_to_next_halving", "is_halving_day", "golden_cross", "cycle_progress_pct",
                  "block_reward_btc", "dxy_close", "us10y_yield"]
OHLC = ["open", "high", "low", "close"]


def stable_id(value):
    return hashlib.sha256(str(value).encode("utf-8")).hexdigest()


def parse_utc(series):
    """Строки дат и Unix-время в секундах/миллисекундах приводятся к UTC."""
    if pd.api.types.is_datetime64_any_dtype(series.dtype):
        # Секундная и наносекундная точность должны совпадать перед concat/merge.
        return pd.to_datetime(series, errors="coerce", utc=True).astype("datetime64[ns, UTC]")
    number = pd.to_numeric(series.dropna(), errors="coerce")
    if len(number) and number.notna().all() and number.abs().median() > 1e8:
        unit = "ms" if number.abs().median() > 1e11 else "s"
        return pd.to_datetime(pd.to_numeric(series, errors="coerce"), unit=unit, errors="coerce", utc=True)
    return pd.to_datetime(series, format="mixed", errors="coerce", utc=True)


def load_sources(data_dir):
    """Читаются только существующие файлы; отсутствующий источник не выдумывается."""
    frames = {}
    for name, filename in FILES.items():
        path = Path(data_dir) / filename
        if not path.is_file():
            continue
        if name in {"market", "news"}:
            frame = pd.read_csv(path, encoding="utf-8-sig", low_memory=False)
        elif name == "intraday":
            with path.open(encoding="utf-8-sig") as stream:
                records = json.load(stream)
            if isinstance(records, dict):
                records = records.get("data", records.get("records", [records]))
            frame = pd.json_normalize(records)
        else:
            root = ET.parse(path).getroot()
            rows = []
            for section in root.findall("section"):
                for record in section.findall("record"):
                    row = {c.tag: " ".join(c.itertext()).strip() for c in record}
                    row["record_id"] = record.get("id")
                    row["record_type"] = row.get("record_type") or section.get("record_type")
                    rows.append(row)
            frame = pd.DataFrame(rows)
        frames[name] = frame
    return frames


def profile_sources(frames, stage):
    """Количественные характеристики и пропуски до/после очистки."""
    rows = []
    for name, frame in frames.items():
        for col in frame:
            rows.append({"stage": stage, "dataset": name, "attribute": col,
                         "rows": len(frame), "columns": len(frame.columns),
                         "memory_mb": frame.memory_usage(deep=True).sum() / 1024**2,
                         "dtype": str(frame[col].dtype), "missing": int(frame[col].isna().sum()),
                         "missing_pct": 100 * frame[col].isna().mean() if len(frame) else np.nan,
                         "unique": frame[col].astype("string").nunique()})
    return pd.DataFrame(rows, columns=["stage", "dataset", "attribute", "rows", "columns",
                                      "memory_mb", "dtype", "missing", "missing_pct", "unique"])


def expand_dictionary(dictionary, frames):
    """Неописанные дополнительные поля помечаются для содержательного уточнения."""
    table = dictionary.copy()
    known = set(zip(table["dataset"], table["attribute"]))
    extra = []
    for name, frame in frames.items():
        for col in frame:
            if (name, col) not in known:
                inherited = table.loc[table["dataset"].eq("market") & table["attribute"].eq(col)]
                if name == "market_daily" and not inherited.empty:
                    row = inherited.iloc[0].to_dict()
                    row["dataset"] = name
                    extra.append(row)
                    continue
                extra.append({"dataset": name, "attribute": col, "data_type": str(frame[col].dtype),
                              "scale": "требует уточнения", "meaning": "Дополнительное поле: уточнить по описанию источника",
                              "unit": "не подтверждена", "analytic_role": "не назначена",
                              "hypothesis_relevance": "Содержательная интерпретация ещё не выполнена"})
    if extra:
        table = pd.concat([table, pd.DataFrame(extra)], ignore_index=True)
    table["present"] = [col in frames.get(name, pd.DataFrame()).columns
                        for name, col in zip(table["dataset"], table["attribute"])]
    return table


In [13]:
dictionary_base = pd.DataFrame([{'dataset': 'market', 'attribute': 'date', 'data_type': 'временной', 'scale': 'интервальная', 'meaning': 'День рыночного наблюдения', 'unit': 'день UTC: подтвердить', 'analytic_role': 'часть ключа', 'hypothesis_relevance': 'Сопоставление наблюдений по времени'}, {'dataset': 'market', 'attribute': 'coin', 'data_type': 'категориальный', 'scale': 'номинальная', 'meaning': 'Обозначение криптовалюты', 'unit': 'тикер', 'analytic_role': 'часть ключа / группа', 'hypothesis_relevance': 'Разделение рынка на активы'}, {'dataset': 'market', 'attribute': 'open', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Цена открытия', 'unit': 'валюта котировки: подтвердить', 'analytic_role': 'состояние объекта', 'hypothesis_relevance': 'Описывает уровень и диапазон цен'}, {'dataset': 'market', 'attribute': 'high', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Максимальная цена дня', 'unit': 'валюта котировки: подтвердить', 'analytic_role': 'состояние объекта', 'hypothesis_relevance': 'Описывает уровень и диапазон цен'}, {'dataset': 'market', 'attribute': 'low', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Минимальная цена дня', 'unit': 'валюта котировки: подтвердить', 'analytic_role': 'состояние объекта', 'hypothesis_relevance': 'Описывает уровень и диапазон цен'}, {'dataset': 'market', 'attribute': 'close', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Цена закрытия', 'unit': 'валюта котировки: подтвердить', 'analytic_role': 'состояние объекта', 'hypothesis_relevance': 'Описывает уровень и диапазон цен'}, {'dataset': 'market', 'attribute': 'volume', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Торговый объём за день', 'unit': 'единица источника: подтвердить', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Торговая активность может быть связана с колебаниями'}, {'dataset': 'market', 'attribute': 'daily_return_pct', 'data_type': 'числовой', 'scale': 'интервальная', 'meaning': 'Простая дневная доходность', 'unit': '%', 'analytic_role': 'результат / состояние', 'hypothesis_relevance': 'Величина и направление изменения цены'}, {'dataset': 'market', 'attribute': 'log_return', 'data_type': 'числовой', 'scale': 'интервальная', 'meaning': 'Логарифмическая дневная доходность', 'unit': 'доли: проверить методику', 'analytic_role': 'результат / состояние', 'hypothesis_relevance': 'Изменение цены, используемое при оценке риска'}, {'dataset': 'market', 'attribute': 'rolling_7d_vol', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Готовый показатель волатильности за 7 дней', 'unit': 'масштаб и формула требуют проверки', 'analytic_role': 'целевой показатель риска', 'hypothesis_relevance': 'Основной признак волатильного состояния'}, {'dataset': 'market', 'attribute': 'rolling_30d_vol', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Готовый показатель волатильности за 30 дней', 'unit': 'масштаб и формула требуют проверки', 'analytic_role': 'целевой показатель риска', 'hypothesis_relevance': 'Основной признак волатильного состояния'}, {'dataset': 'market', 'attribute': 'ath_usd', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Исторический максимум, указанный источником', 'unit': 'USD по имени поля', 'analytic_role': 'контекст состояния', 'hypothesis_relevance': 'База расчёта просадки; проверить историческую доступность'}, {'dataset': 'market', 'attribute': 'drawdown_from_ath_pct', 'data_type': 'числовой', 'scale': 'интервальная', 'meaning': 'Отклонение от указанного исторического максимума', 'unit': '%', 'analytic_role': 'состояние объекта', 'hypothesis_relevance': 'Характеризует глубину снижения'}, {'dataset': 'market', 'attribute': 'sma_50d', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Скользящая средняя цены на окне, указанном в имени', 'unit': 'валюта котировки', 'analytic_role': 'технический контекст', 'hypothesis_relevance': 'Описывает тренд; связан с исходной ценой по построению'}, {'dataset': 'market', 'attribute': 'sma_200d', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Скользящая средняя цены на окне, указанном в имени', 'unit': 'валюта котировки', 'analytic_role': 'технический контекст', 'hypothesis_relevance': 'Описывает тренд; связан с исходной ценой по построению'}, {'dataset': 'market', 'attribute': 'golden_cross', 'data_type': 'категориальный бинарный', 'scale': 'номинальная', 'meaning': 'Флаг пересечения средних или состояния тренда: уточнить', 'unit': '0/1', 'analytic_role': 'технический контекст', 'hypothesis_relevance': 'Группа рыночного состояния; не независимая причина'}, {'dataset': 'market', 'attribute': 'fear_greed_value', 'data_type': 'числовой индекс', 'scale': 'порядковая; интервальность не подтверждена', 'meaning': 'Уровень страха и жадности', 'unit': 'баллы 0–100', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Проверка связи настроений и волатильности; проверить состав индекса'}, {'dataset': 'market', 'attribute': 'fear_greed_label', 'data_type': 'категориальный', 'scale': 'порядковая: подтвердить порядок', 'meaning': 'Группа рыночных настроений', 'unit': 'метка', 'analytic_role': 'группировочный признак', 'hypothesis_relevance': 'Категориальное представление индекса; может дублировать value'}, {'dataset': 'market', 'attribute': 'fear_greed_zone', 'data_type': 'категориальный', 'scale': 'порядковая: подтвердить порядок', 'meaning': 'Группа рыночных настроений', 'unit': 'метка', 'analytic_role': 'группировочный признак', 'hypothesis_relevance': 'Категориальное представление индекса; может дублировать value'}, {'dataset': 'market', 'attribute': 'btc_dominance_pct', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Доля соответствующего актива в общей капитализации', 'unit': '%', 'analytic_role': 'рыночный контекст / фактор', 'hypothesis_relevance': 'Характеризует структуру рынка'}, {'dataset': 'market', 'attribute': 'eth_dominance_pct', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Доля соответствующего актива в общей капитализации', 'unit': '%', 'analytic_role': 'рыночный контекст / фактор', 'hypothesis_relevance': 'Характеризует структуру рынка'}, {'dataset': 'market', 'attribute': 'total_market_cap_usd', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Общая или соответствующая активу капитализация', 'unit': 'USD', 'analytic_role': 'рыночный контекст', 'hypothesis_relevance': 'Масштаб рынка; показатели взаимосвязаны по определению'}, {'dataset': 'market', 'attribute': 'btc_market_cap_usd', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Общая или соответствующая активу капитализация', 'unit': 'USD', 'analytic_role': 'рыночный контекст', 'hypothesis_relevance': 'Масштаб рынка; показатели взаимосвязаны по определению'}, {'dataset': 'market', 'attribute': 'eth_market_cap_usd', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Общая или соответствующая активу капитализация', 'unit': 'USD', 'analytic_role': 'рыночный контекст', 'hypothesis_relevance': 'Масштаб рынка; показатели взаимосвязаны по определению'}, {'dataset': 'market', 'attribute': 'halving_epoch', 'data_type': 'категориальный', 'scale': 'порядковая', 'meaning': 'Эпоха халвинга Bitcoin', 'unit': 'номер эпохи', 'analytic_role': 'контекст цикла', 'hypothesis_relevance': 'Сопоставление периодов; эпоха смешана с календарным временем'}, {'dataset': 'market', 'attribute': 'days_since_halving', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число дней после халвинга Bitcoin', 'unit': 'дни', 'analytic_role': 'контекст цикла', 'hypothesis_relevance': 'Положение наблюдения внутри цикла'}, {'dataset': 'market', 'attribute': 'days_to_next_halving', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Дни до следующего халвинга по расчёту источника', 'unit': 'дни', 'analytic_role': 'описательный контекст', 'hypothesis_relevance': 'Будущая фактическая дата могла быть неизвестна: не готовый предиктор'}, {'dataset': 'market', 'attribute': 'is_halving_day', 'data_type': 'категориальный бинарный', 'scale': 'номинальная', 'meaning': 'Флаг дня халвинга Bitcoin', 'unit': '0/1', 'analytic_role': 'контекст события', 'hypothesis_relevance': 'Выделение дат события'}, {'dataset': 'market', 'attribute': 'cycle_progress_pct', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Доля прошедшего цикла по методике источника', 'unit': '%', 'analytic_role': 'описательный контекст', 'hypothesis_relevance': 'Может использовать будущую дату; проверить до прогнозирования'}, {'dataset': 'market', 'attribute': 'block_reward_btc', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Награда за блок Bitcoin', 'unit': 'BTC', 'analytic_role': 'контекст предложения', 'hypothesis_relevance': 'Описывает параметры протокола и цикл'}, {'dataset': 'market', 'attribute': 'dxy_close', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Значение индекса доллара', 'unit': 'пункты', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Внешний макроэкономический фон'}, {'dataset': 'market', 'attribute': 'us10y_yield', 'data_type': 'числовой', 'scale': 'интервальная', 'meaning': 'Доходность 10-летних облигаций США', 'unit': '% годовых: подтвердить', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Внешний процентный фон'}, {'dataset': 'intraday', 'attribute': 'timestamp', 'data_type': 'временной', 'scale': 'интервальная', 'meaning': 'Время начала/конца четырёхчасовой свечи', 'unit': 'UTC: трактовка задаётся параметром', 'analytic_role': 'ключ / время', 'hypothesis_relevance': 'Приведение JSON к дневному уровню'}, {'dataset': 'intraday', 'attribute': 'open', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Цена открытия 4h', 'unit': 'USDT по имени файла: подтвердить', 'analytic_role': 'состояние BTC', 'hypothesis_relevance': 'Дополнительное измерение внутридневной динамики'}, {'dataset': 'intraday', 'attribute': 'high', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Максимальная цена 4h', 'unit': 'USDT по имени файла: подтвердить', 'analytic_role': 'состояние BTC', 'hypothesis_relevance': 'Дополнительное измерение внутридневной динамики'}, {'dataset': 'intraday', 'attribute': 'low', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Минимальная цена 4h', 'unit': 'USDT по имени файла: подтвердить', 'analytic_role': 'состояние BTC', 'hypothesis_relevance': 'Дополнительное измерение внутридневной динамики'}, {'dataset': 'intraday', 'attribute': 'close', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Цена закрытия 4h', 'unit': 'USDT по имени файла: подтвердить', 'analytic_role': 'состояние BTC', 'hypothesis_relevance': 'Дополнительное измерение внутридневной динамики'}, {'dataset': 'intraday', 'attribute': 'Fear & Greed Index', 'data_type': 'числовой индекс', 'scale': 'порядковая; интервальность не подтверждена', 'meaning': 'Индекс рыночных настроений', 'unit': 'баллы 0–100', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Сопоставление настроений из другого источника'}, {'dataset': 'intraday', 'attribute': 'Fear & Greed Classification', 'data_type': 'категориальный', 'scale': 'порядковая', 'meaning': 'Категория индекса настроений', 'unit': 'метка', 'analytic_role': 'группа настроений', 'hypothesis_relevance': 'Описательное сравнение групп'}, {'dataset': 'news', 'attribute': 'url', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Адрес публикации', 'unit': 'строка', 'analytic_role': 'естественный идентификатор', 'hypothesis_relevance': 'Источник и идентификация; thumbnail не используется в проверке гипотезы'}, {'dataset': 'news', 'attribute': 'source', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Источник публикации', 'unit': 'строка', 'analytic_role': 'контекст выборки', 'hypothesis_relevance': 'Источник и идентификация; thumbnail не используется в проверке гипотезы'}, {'dataset': 'news', 'attribute': 'thumbnail', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Ссылка на иллюстрацию', 'unit': 'строка', 'analytic_role': 'служебный атрибут', 'hypothesis_relevance': 'Источник и идентификация; thumbnail не используется в проверке гипотезы'}, {'dataset': 'news', 'attribute': 'title', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Заголовок или описание публикации', 'unit': 'текст', 'analytic_role': 'неструктурированный факторный источник', 'hypothesis_relevance': 'Аннотирование и TF-IDF; не готовая оценка влияния на цену'}, {'dataset': 'news', 'attribute': 'description', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Заголовок или описание публикации', 'unit': 'текст', 'analytic_role': 'неструктурированный факторный источник', 'hypothesis_relevance': 'Аннотирование и TF-IDF; не готовая оценка влияния на цену'}, {'dataset': 'news', 'attribute': 'createdAt', 'data_type': 'временной', 'scale': 'интервальная', 'meaning': 'Время публикации', 'unit': 'UTC', 'analytic_role': 'время / агрегирование', 'hypothesis_relevance': 'Дневные признаки информационного фона'}, {'dataset': 'news', 'attribute': 'news_id', 'data_type': 'идентификатор', 'scale': 'номинальная', 'meaning': 'SHA-256 от URL или текста с временем', 'unit': 'строка', 'analytic_role': 'ключ обновления', 'hypothesis_relevance': 'Служебный ключ, не фактор рынка'}, {'dataset': 'bots', 'attribute': 'record_id', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Идентификатор записи', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'record_type', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Тип записи', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'bot_category', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Категория торгового бота', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'chain', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Поддерживаемые блокчейны', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'section', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Раздел справочника', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'model', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Модель, сформировавшая ответ', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'provider', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Поставщик модели', 'unit': 'метка', 'analytic_role': 'справочный контекст', 'hypothesis_relevance': 'Нет временного ключа активности; прямое влияние не оценивается'}, {'dataset': 'bots', 'attribute': 'question', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Вопрос или ответ / описание инструмента', 'unit': 'текст', 'analytic_role': 'справочный текст', 'hypothesis_relevance': 'Аннотирование и векторизация; ответы моделей требуют внешней проверки'}, {'dataset': 'bots', 'attribute': 'answer', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Вопрос или ответ / описание инструмента', 'unit': 'текст', 'analytic_role': 'справочный текст', 'hypothesis_relevance': 'Аннотирование и векторизация; ответы моделей требуют внешней проверки'}, {'dataset': 'market_daily', 'attribute': 'json_open', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Первая цена открытия в полных сутках JSON', 'unit': 'валюта JSON', 'analytic_role': 'дополнительное состояние BTC', 'hypothesis_relevance': 'Сопоставление с CSV без подмены его цены'}, {'dataset': 'market_daily', 'attribute': 'json_high', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Максимальная цена в полных сутках JSON', 'unit': 'валюта JSON', 'analytic_role': 'дополнительное состояние BTC', 'hypothesis_relevance': 'Сопоставление с CSV без подмены его цены'}, {'dataset': 'market_daily', 'attribute': 'json_low', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Минимальная цена в полных сутках JSON', 'unit': 'валюта JSON', 'analytic_role': 'дополнительное состояние BTC', 'hypothesis_relevance': 'Сопоставление с CSV без подмены его цены'}, {'dataset': 'market_daily', 'attribute': 'json_close', 'data_type': 'числовой', 'scale': 'отношений', 'meaning': 'Последняя цена закрытия в полных сутках JSON', 'unit': 'валюта JSON', 'analytic_role': 'дополнительное состояние BTC', 'hypothesis_relevance': 'Сопоставление с CSV без подмены его цены'}, {'dataset': 'market_daily', 'attribute': 'json_bars', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число свечей / разных значений индекса за сутки', 'unit': 'количество', 'analytic_role': 'контроль качества', 'hypothesis_relevance': 'Оценка полноты и согласованности'}, {'dataset': 'market_daily', 'attribute': 'json_fg_distinct', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число свечей / разных значений индекса за сутки', 'unit': 'количество', 'analytic_role': 'контроль качества', 'hypothesis_relevance': 'Оценка полноты и согласованности'}, {'dataset': 'market_daily', 'attribute': 'json_complete', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Полнота суток / наличие совпадения с источником', 'unit': 'True/False', 'analytic_role': 'контроль покрытия', 'hypothesis_relevance': 'Отличает доступность данных от значения признака'}, {'dataset': 'market_daily', 'attribute': 'json_matched', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Полнота суток / наличие совпадения с источником', 'unit': 'True/False', 'analytic_role': 'контроль покрытия', 'hypothesis_relevance': 'Отличает доступность данных от значения признака'}, {'dataset': 'market_daily', 'attribute': 'news_matched', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Полнота суток / наличие совпадения с источником', 'unit': 'True/False', 'analytic_role': 'контроль покрытия', 'hypothesis_relevance': 'Отличает доступность данных от значения признака'}, {'dataset': 'market_daily', 'attribute': 'json_fear_greed', 'data_type': 'числовой индекс', 'scale': 'порядковая', 'meaning': 'Последний доступный индекс из JSON за день', 'unit': 'баллы 0–100', 'analytic_role': 'кандидат-фактор', 'hypothesis_relevance': 'Отдельный источник; не заполняет автоматически индекс CSV'}, {'dataset': 'market_daily', 'attribute': 'news_count', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число публикаций/тематических меток/источников в корпусе за день', 'unit': 'количество', 'analytic_role': 'кандидат-фактор / контекст', 'hypothesis_relevance': 'Измеримый информационный фон в пределах корпуса, а не всего рынка'}, {'dataset': 'market_daily', 'attribute': 'news_security_count', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число публикаций/тематических меток/источников в корпусе за день', 'unit': 'количество', 'analytic_role': 'кандидат-фактор / контекст', 'hypothesis_relevance': 'Измеримый информационный фон в пределах корпуса, а не всего рынка'}, {'dataset': 'market_daily', 'attribute': 'news_regulation_count', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число публикаций/тематических меток/источников в корпусе за день', 'unit': 'количество', 'analytic_role': 'кандидат-фактор / контекст', 'hypothesis_relevance': 'Измеримый информационный фон в пределах корпуса, а не всего рынка'}, {'dataset': 'market_daily', 'attribute': 'news_adoption_count', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число публикаций/тематических меток/источников в корпусе за день', 'unit': 'количество', 'analytic_role': 'кандидат-фактор / контекст', 'hypothesis_relevance': 'Измеримый информационный фон в пределах корпуса, а не всего рынка'}, {'dataset': 'market_daily', 'attribute': 'news_sources', 'data_type': 'числовой дискретный', 'scale': 'отношений', 'meaning': 'Число публикаций/тематических меток/источников в корпусе за день', 'unit': 'количество', 'analytic_role': 'кандидат-фактор / контекст', 'hypothesis_relevance': 'Измеримый информационный фон в пределах корпуса, а не всего рынка'}, {'dataset': 'market_daily', 'attribute': 'year', 'data_type': 'категориальный', 'scale': 'порядковая', 'meaning': 'Год наблюдения для разделения Parquet', 'unit': 'год', 'analytic_role': 'физическая организация хранения', 'hypothesis_relevance': 'Не самостоятельный рыночный фактор'}, {'dataset': 'documents', 'attribute': 'doc_id', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Идентификатор документа с префиксом источника', 'unit': 'строка', 'analytic_role': 'ключ', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'source_record_id', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Ключ записи исходного источника', 'unit': 'строка', 'analytic_role': 'связь со справочником/новостью', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'source_kind', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Источник текста: news или bots', 'unit': 'строка', 'analytic_role': 'группа', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'source', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Издание или XML-справочник', 'unit': 'строка', 'analytic_role': 'контекст', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'topic', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Одна или несколько тем по словарным правилам', 'unit': 'строка', 'analytic_role': 'автоматическая аннотация', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'coin_mentions', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Распознанные упоминания монет', 'unit': 'строка', 'analytic_role': 'автоматическая аннотация', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'annotation_method', 'data_type': 'категориальный / идентификатор', 'scale': 'номинальная', 'meaning': 'Версия правил аннотирования', 'unit': 'строка', 'analytic_role': 'метаданные', 'hypothesis_relevance': 'Позволяет интерпретировать и проследить текстовые признаки'}, {'dataset': 'documents', 'attribute': 'text', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Исходный объединённый или нормализованный текст', 'unit': 'текст', 'analytic_role': 'корпус для векторизации', 'hypothesis_relevance': 'Исходный смысл сохраняется; нормализация используется для признаков'}, {'dataset': 'documents', 'attribute': 'text_clean', 'data_type': 'текстовый', 'scale': 'не применяется к свободному тексту', 'meaning': 'Исходный объединённый или нормализованный текст', 'unit': 'текст', 'analytic_role': 'корпус для векторизации', 'hypothesis_relevance': 'Исходный смысл сохраняется; нормализация используется для признаков'}, {'dataset': 'documents', 'attribute': 'date', 'data_type': 'временной', 'scale': 'интервальная', 'meaning': 'День новости; у XML отсутствует', 'unit': 'день UTC', 'analytic_role': 'время / связь', 'hypothesis_relevance': 'XML не получает вымышленную дату'}, {'dataset': 'documents', 'attribute': 'is_security', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Наличие ключевых слов соответствующей темы', 'unit': 'True/False', 'analytic_role': 'предварительная аннотация', 'hypothesis_relevance': 'Служит агрегации; требует ручной проверки, не является тональностью'}, {'dataset': 'documents', 'attribute': 'is_regulation', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Наличие ключевых слов соответствующей темы', 'unit': 'True/False', 'analytic_role': 'предварительная аннотация', 'hypothesis_relevance': 'Служит агрегации; требует ручной проверки, не является тональностью'}, {'dataset': 'documents', 'attribute': 'is_adoption', 'data_type': 'бинарный', 'scale': 'номинальная', 'meaning': 'Наличие ключевых слов соответствующей темы', 'unit': 'True/False', 'analytic_role': 'предварительная аннотация', 'hypothesis_relevance': 'Служит агрегации; требует ручной проверки, не является тональностью'}, {'dataset': 'documents', 'attribute': 'tfidf_row', 'data_type': 'индекс', 'scale': 'номинальная', 'meaning': 'Номер строки документа в разреженной матрице', 'unit': 'индекс с нуля', 'analytic_role': 'техническая связь', 'hypothesis_relevance': 'Сохраняет соответствие документа и вектора'}, {'dataset': 'text_tfidf', 'attribute': 'term_0 … term_N', 'data_type': 'числовой', 'scale': 'отношений весов в фиксированной модели', 'meaning': 'Нормированный вес термина/биграммы TF-IDF', 'unit': 'безразмерный вес', 'analytic_role': 'векторный признак', 'hypothesis_relevance': 'Лексическое представление текста; не сила причинного влияния на рынок'}])

# Читаем источники заново: очистка должна учитывать исходные, ещё не преобразованные значения.
raw_23 = load_sources(DATA_DIR)
attribute_dictionary = expand_dictionary(dictionary_base, raw_23)
for source in FILES:
    print(f"\nСловарь атрибутов: {source}")
    columns = ["attribute", "data_type", "scale", "meaning", "unit", "analytic_role", "hypothesis_relevance", "present"]
    display(attribute_dictionary.loc[attribute_dictionary["dataset"].eq(source), columns])



Словарь атрибутов: market


,attribute,data_type,scale,meaning,unit,analytic_role,hypothesis_relevance,present
0,date,временной,интервальная,День рыночного наблюдения,день UTC: подтвердить,часть ключа,Сопоставление наблюдений по времени,True
1,coin,категориальный,номинальная,Обозначение криптовалюты,тикер,часть ключа / группа,Разделение рынка на активы,True
2,open,числовой,отношений,Цена открытия,валюта котировки: подтвердить,состояние объекта,Описывает уровень и диапазон цен,True
3,high,числовой,отношений,Максимальная цена дня,валюта котировки: подтвердить,состояние объекта,Описывает уровень и диапазон цен,True
4,low,числовой,отношений,Минимальная цена дня,валюта котировки: подтвердить,состояние объекта,Описывает уровень и диапазон цен,True
5,close,числовой,отношений,Цена закрытия,валюта котировки: подтвердить,состояние объекта,Описывает уровень и диапазон цен,True
6,volume,числовой,отношений,Торговый объём за день,единица источника: подтвердить,кандидат-фактор,Торговая активность может быть связана с колеб...,True
7,daily_return_pct,числовой,интервальная,Простая дневная доходность,%,результат / состояние,Величина и направление изменения цены,True
8,log_return,числовой,интервальная,Логарифмическая дневная доходность,доли: проверить методику,результат / состояние,"Изменение цены, используемое при оценке риска",True
9,rolling_7d_vol,числовой,отношений,Готовый показатель волатильности за 7 дней,масштаб и формула требуют проверки,целевой показатель риска,Основной признак волатильного состояния,True



Словарь атрибутов: intraday


,attribute,data_type,scale,meaning,unit,analytic_role,hypothesis_relevance,present
32,timestamp,временной,интервальная,Время начала/конца четырёхчасовой свечи,UTC: трактовка задаётся параметром,ключ / время,Приведение JSON к дневному уровню,True
33,open,числовой,отношений,Цена открытия 4h,USDT по имени файла: подтвердить,состояние BTC,Дополнительное измерение внутридневной динамики,True
34,high,числовой,отношений,Максимальная цена 4h,USDT по имени файла: подтвердить,состояние BTC,Дополнительное измерение внутридневной динамики,True
35,low,числовой,отношений,Минимальная цена 4h,USDT по имени файла: подтвердить,состояние BTC,Дополнительное измерение внутридневной динамики,True
36,close,числовой,отношений,Цена закрытия 4h,USDT по имени файла: подтвердить,состояние BTC,Дополнительное измерение внутридневной динамики,True
37,Fear & Greed Index,числовой индекс,порядковая; интервальность не подтверждена,Индекс рыночных настроений,баллы 0–100,кандидат-фактор,Сопоставление настроений из другого источника,True
38,Fear & Greed Classification,категориальный,порядковая,Категория индекса настроений,метка,группа настроений,Описательное сравнение групп,True



Словарь атрибутов: news


,attribute,data_type,scale,meaning,unit,analytic_role,hypothesis_relevance,present
39,url,категориальный / идентификатор,номинальная,Адрес публикации,строка,естественный идентификатор,Источник и идентификация; thumbnail не использ...,True
40,source,категориальный / идентификатор,номинальная,Источник публикации,строка,контекст выборки,Источник и идентификация; thumbnail не использ...,True
41,thumbnail,категориальный / идентификатор,номинальная,Ссылка на иллюстрацию,строка,служебный атрибут,Источник и идентификация; thumbnail не использ...,True
42,title,текстовый,не применяется к свободному тексту,Заголовок или описание публикации,текст,неструктурированный факторный источник,Аннотирование и TF-IDF; не готовая оценка влия...,True
43,description,текстовый,не применяется к свободному тексту,Заголовок или описание публикации,текст,неструктурированный факторный источник,Аннотирование и TF-IDF; не готовая оценка влия...,True
44,createdAt,временной,интервальная,Время публикации,UTC,время / агрегирование,Дневные признаки информационного фона,True
45,news_id,идентификатор,номинальная,SHA-256 от URL или текста с временем,строка,ключ обновления,"Служебный ключ, не фактор рынка",False



Словарь атрибутов: bots


,attribute,data_type,scale,meaning,unit,analytic_role,hypothesis_relevance,present
46,record_id,категориальный / идентификатор,номинальная,Идентификатор записи,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
47,record_type,категориальный / идентификатор,номинальная,Тип записи,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
48,bot_category,категориальный / идентификатор,номинальная,Категория торгового бота,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
49,chain,категориальный / идентификатор,номинальная,Поддерживаемые блокчейны,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
50,section,категориальный / идентификатор,номинальная,Раздел справочника,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
51,model,категориальный / идентификатор,номинальная,"Модель, сформировавшая ответ",метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
52,provider,категориальный / идентификатор,номинальная,Поставщик модели,метка,справочный контекст,Нет временного ключа активности; прямое влияни...,True
53,question,текстовый,не применяется к свободному тексту,Вопрос или ответ / описание инструмента,текст,справочный текст,Аннотирование и векторизация; ответы моделей т...,True
54,answer,текстовый,не применяется к свободному тексту,Вопрос или ответ / описание инструмента,текст,справочный текст,Аннотирование и векторизация; ответы моделей т...,True
86,source_url,object,требует уточнения,Дополнительное поле: уточнить по описанию исто...,не подтверждена,не назначена,Содержательная интерпретация ещё не выполнена,True


## 2.3.2. Очистка и протокол изменений

| Проверка | Действие и обоснование |
|---|---|
| Пустые строки, числовые строки и даты | Приведение к единому представлению; ошибки преобразования фиксируются |
| Полные повторы | Удаляются повторные строки, чтобы не удваивать наблюдения |
| Нет ключа, времени или текста новости | Запись откладывается в `rejected` для проверки |
| Различающиеся записи с одним ключом внутри входного пакета | Все варианты откладываются, правильный вариант не выбирается случайно |
| Цена ≤ 0, отрицательный объём, индекс вне 0–100, нарушение OHLC | Ошибочные значения заменяются пропусками; количество изменений записывается |
| Пропуски цен, доходностей и скользящих показателей | Сохраняются; ноль не подставляется |
| Сильные, но логически допустимые движения цены | Сохраняются: они могут быть предметом исследования |

Ключи: CSV рынка — `coin + date`, JSON — `timestamp`, новости — `news_id`, XML — `record_id`.
`news_id` строится по URL; если ссылки нет — по тексту и времени. Изменение текста
без URL может создать новый ключ; это ограничение нужно учитывать при обновлении.
У XML без ID используется хеш содержимого с тем же ограничением.


In [14]:
def clean_sources(raw, timestamp_kind="open"):
    """Полные повторы удаляются; конфликтующие ключи отправляются на проверку."""
    if timestamp_kind not in {"open", "close"}:
        raise ValueError("timestamp_kind должен быть open или close")
    cleaned, rejected, audit = {}, [], []
    required = {"market": ["coin", "date", "close"], "intraday": ["timestamp"] + OHLC,
                "news": ["createdAt"], "bots": ["record_type"]}
    for name, source in raw.items():
        frame = source.copy()
        if frame.empty:
            cleaned[name] = frame
            continue
        frame.columns = frame.columns.astype(str).str.strip()
        absent = set(required[name]) - set(frame.columns)
        if absent or (name == "news" and not {"title", "description"}.intersection(frame.columns)):
            raise ValueError(f"{name}: не хватает обязательных полей {sorted(absent)} или текста новости")
        def note(rule, count):
            audit.append({"dataset": name, "rule": rule, "count": int(count)})
        def reject(mask, reason):
            bad = frame.loc[mask].copy()
            if not bad.empty:
                bad["dataset"], bad["reason"] = name, reason
                rejected.append(bad)
            note(reason, mask.sum())
        for col in frame.select_dtypes(include=["object", "string"]):
            frame[col] = frame[col].map(lambda x: json.dumps(x, ensure_ascii=False)
                                       if isinstance(x, (list, dict)) else x)
            frame[col] = frame[col].astype("string").str.strip().replace("", pd.NA)
        date_col = {"market": "date", "intraday": "timestamp", "news": "createdAt"}.get(name)
        if date_col:
            converted = parse_utc(frame[date_col])
            note(f"Не распознана дата {date_col}", (frame[date_col].notna() & converted.isna()).sum())
            frame[date_col] = converted.dt.normalize() if name == "market" else converted
        numeric = MARKET_NUMERIC if name == "market" else OHLC + ["Fear & Greed Index"] if name == "intraday" else []
        for col in numeric:
            if col in frame:
                converted = pd.to_numeric(frame[col], errors="coerce").astype(float).replace([np.inf, -np.inf], np.nan)
                note(f"Не распознано число {col}", (frame[col].notna() & converted.isna()).sum())
                frame[col] = converted
        if name == "market":
            frame["coin"] = frame["coin"].astype("string").str.upper()
        if name in {"market", "intraday"}:
            for col in [c for c in OHLC if c in frame]:
                bad = frame[col].le(0)
                note(f"{col} <= 0: заменено пропуском", bad.sum())
                frame.loc[bad, col] = np.nan
            if all(c in frame for c in OHLC):
                bad = (frame["high"].lt(frame["low"]) | frame["open"].gt(frame["high"]) |
                       frame["open"].lt(frame["low"]) | frame["close"].gt(frame["high"]) | frame["close"].lt(frame["low"]))
                note("Нарушение OHLC: цены заменены пропусками", bad.sum())
                frame.loc[bad, OHLC] = np.nan
            for col in ["volume", "rolling_7d_vol", "rolling_30d_vol", "total_market_cap_usd",
                        "btc_market_cap_usd", "eth_market_cap_usd"]:
                if col in frame:
                    bad = frame[col].lt(0)
                    note(f"{col} < 0: заменено пропуском", bad.sum())
                    frame.loc[bad, col] = np.nan
            for col in ["fear_greed_value", "Fear & Greed Index", "btc_dominance_pct", "eth_dominance_pct", "cycle_progress_pct"]:
                if col in frame:
                    bad = frame[col].notna() & ~frame[col].between(0, 100)
                    note(f"{col} вне [0,100]: заменено пропуском", bad.sum())
                    frame.loc[bad, col] = np.nan
            for col in ["golden_cross", "is_halving_day"]:
                if col in frame:
                    bad = frame[col].notna() & ~frame[col].isin([0, 1])
                    note(f"{col} не 0/1: заменено пропуском", bad.sum())
                    frame.loc[bad, col] = np.nan
        if name == "news":
            for col in ["title", "description", "url", "source"]:
                if col not in frame:
                    frame[col] = pd.NA
            text = frame["title"].fillna("") + " " + frame["description"].fillna("")
            identity = frame["url"].fillna("")
            fallback = text.str.casefold().str.strip() + "|" + frame["createdAt"].astype(str)
            frame["news_id"] = ("url:" + identity).where(frame["url"].notna(), "text:" + fallback).map(stable_id)
            bad = frame["createdAt"].isna() | text.str.strip().eq("")
        elif name == "bots":
            if "record_id" not in frame:
                frame["record_id"] = pd.NA
            fallback = frame.drop(columns="record_id").astype("string").fillna("").agg("|".join, axis=1).map(stable_id)
            frame["record_id"] = frame["record_id"].fillna(fallback)
            bad = frame["record_id"].isna()
        else:
            bad = frame[KEYS[name]].isna().any(axis=1)
        reject(bad, "Нет ключа, времени или текста")
        frame = frame.loc[~bad].copy()
        if name == "intraday":
            start = frame["timestamp"] - (pd.Timedelta(hours=4) if timestamp_kind == "close" else pd.Timedelta(0))
            off_grid = start.ne(start.dt.floor("4h"))
            reject(off_grid, "Время вне сетки 4h: требуется уточнение")
            frame = frame.loc[~off_grid].copy()
        exact = frame.astype("string").duplicated()
        note("Полные повторы: удалены повторные строки", exact.sum())
        frame = frame.loc[~exact].copy()
        conflicts = frame.duplicated(KEYS[name], keep=False)
        reject(conflicts, "Различающиеся записи с одним ключом")
        frame = frame.loc[~conflicts].sort_values(KEYS[name]).reset_index(drop=True)
        cleaned[name] = frame
    rejected_frame = pd.concat(rejected, ignore_index=True, sort=False) if rejected else pd.DataFrame(columns=["dataset", "reason"])
    return cleaned, rejected_frame, pd.DataFrame(audit, columns=["dataset", "rule", "count"])


## 2.3.3. Аннотирование и векторизация неструктурированных данных

В выбранных файлах есть **текст**: заголовки/описания новостей и вопросы/ответы XML.
Аудио и видео в предоставленном описании отсутствуют, поэтому их обработка не требуется.

Аннотирование — добавление меток. Используются три простые темы: `security`
(безопасность), `regulation` (регулирование), `adoption` (принятие/партнёрства).
Тем может быть несколько; `other` означает, что правило ничего не распознало.
Отдельно отмечаются упоминания BTC, ETH, SOL, ADA, XRP.
Это словарная автоматическая разметка, **не оценка тональности** и не замена ручной проверке.
Проверка выборки из 20 текстов предусмотрена ниже. Ирония, отрицания, синонимы и
неоднозначные тикеры могут приводить к ошибкам.

Векторизация выполняется TF-IDF: каждому тексту соответствует вектор весов слов и
сочетаний из двух слов. Ограничение — 2000 признаков, английские служебные слова
исключаются, кириллица сохраняется. Разреженная матрица хранится отдельно, а `doc_id`
и `tfidf_row` связывают вектор с документом. Веса не интерпретируются как влияние на цену.
Сейчас весь корпус векторизуется для описания; для будущего прогноза словарь нужно
обучать только на обучающем периоде.


In [15]:
def annotate_texts(cleaned):
    """Правила дают предварительные тематические метки, а не экспертную тональность."""
    parts = []
    for name, text_columns, key in [("news", ["title", "description"], "news_id"),
                                    ("bots", ["question", "answer"], "record_id")]:
        frame = cleaned.get(name, pd.DataFrame())
        if frame.empty:
            continue
        texts = pd.Series("", index=frame.index, dtype="string")
        for col in text_columns:
            if col in frame:
                texts = texts + " " + frame[col].fillna("").astype("string")
        part = pd.DataFrame({"doc_id": name + ":" + frame[key].astype(str), "source_kind": name,
                             "source_record_id": frame[key], "text": texts.str.strip()})
        part["date"] = frame["createdAt"].dt.normalize() if name == "news" else pd.NaT
        part["source"] = frame["source"] if name == "news" else "xml_reference"
        parts.append(part.loc[part["text"].str.len().gt(0)])
    columns = ["doc_id", "source_kind", "source_record_id", "text", "date", "source"]
    docs = pd.concat(parts, ignore_index=True) if parts else pd.DataFrame(columns=columns)
    docs["date"] = pd.to_datetime(docs["date"], utc=True)
    docs["text_clean"] = docs["text"].astype("string").str.casefold().str.replace(r"\s+", " ", regex=True).str.strip()
    patterns = {
        "security": r"\b(?:hack\w*|exploit\w*|scam\w*|fraud\w*|взлом\w*|мошеннич\w*)\b",
        "regulation": r"\b(?:regulat\w*|lawsuit\w*|ban|banned|запрет\w*|регулир\w*)\b",
        "adoption": r"\b(?:adoption|approval|approved|partnership\w*|одобрен\w*|партн[её]рств\w*)\b",
    }
    for topic, pattern in patterns.items():
        docs[f"is_{topic}"] = docs["text_clean"].str.contains(pattern, regex=True, na=False).astype(bool)
    docs["topic"] = docs.apply(lambda row: "|".join(t for t in patterns if row[f"is_{t}"]) or "other", axis=1) if len(docs) else pd.Series(dtype="string")
    coins = {"BTC": r"\b(?:bitcoin|btc|биткоин\w*)\b", "ETH": r"\b(?:ethereum|eth|эфириум\w*)\b",
             "SOL": r"\b(?:solana|sol)\b", "ADA": r"\b(?:cardano|ada)\b", "XRP": r"\b(?:ripple|xrp)\b"}
    docs["coin_mentions"] = docs["text_clean"].map(lambda text: "|".join(c for c, p in coins.items() if re.search(p, text)))
    docs["annotation_method"] = "keywords_v1"
    docs = docs.sort_values("doc_id").reset_index(drop=True)
    docs["tfidf_row"] = np.arange(len(docs))
    return docs


def vectorize_texts(documents):
    """Одна строка разреженной TF-IDF-матрицы соответствует documents.tfidf_row."""
    vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), lowercase=False,
                                 stop_words="english", dtype=np.float32)
    if documents.empty:
        return sparse.csr_matrix((0, 0), dtype=np.float32), [], np.array([])
    try:
        matrix = vectorizer.fit_transform(documents["text_clean"])
    except ValueError as exc:
        if "empty vocabulary" not in str(exc):
            raise
        return sparse.csr_matrix((len(documents), 0), dtype=np.float32), [], np.array([])
    return matrix, vectorizer.get_feature_names_out().tolist(), vectorizer.idf_


## 2.3.4. Общие атрибуты и правила слияния

Итоговая аналитическая единица — **монета × день UTC**. Базой остаётся дневной CSV.

| Источник | Общие атрибуты / агрегаты | Правило |
|---|---|---|
| Дневной CSV | `date`, `coin` | Основная таблица; одна строка на ключ |
| JSON BTC 4h | `date` из timestamp, `coin = BTC`; дневные OHLC, число свечей, последний индекс | Шесть корректных свечей образуют полные сутки; соединение по `date + coin` |
| Новости | `date` из `createdAt`; число публикаций, тематических меток и источников | Общерыночные дневные показатели; соединение по `date` |
| XML | `record_id`, категория, блокчейн; ссылка текста через `source_record_id` | Справочник в том же хранилище; у записей нет подтверждённого временного ключа |

По умолчанию JSON timestamp считается **началом** свечи (`TIMESTAMP_KIND = "open"`).
Если это конец, установите `close`; для формата `03:59:59.999` потребуется отдельное
уточнение, такие времена не округляются молча. Наивные даты принимаются за UTC — проверить у источника.
Для неполных суток JSON дневные цены остаются пустыми. Если индекс внутри суток
меняется, это видно по `json_fg_distinct`; дневное значение — последнее доступное.

Используется `left merge`, чтобы сохранить все очищенные строки основного CSV.
Кардинальность соединения проверяется, размножение строк не допускается.
Цена CSV и `json_close` остаются разными атрибутами: площадки и USD/USDT могут различаться.
Отсутствие новостей за дату остаётся пропуском: неполная выгрузка не доказывает нулевой поток.
Одни и те же общерыночные новости повторяются у монет одного дня и не являются
независимыми событиями для каждой монеты.

XML включён в единую структуру данных как справочник и текстовый корпус.
Присоединение его строк ко всем ценам создало бы вымышленную связь и дубликаты.


In [16]:
def aggregate_intraday(frame, timestamp_kind="open"):
    columns = ["date", "coin", "json_bars", "json_complete", "json_open", "json_high", "json_low", "json_close", "json_fear_greed", "json_fg_distinct"]
    if frame.empty:
        return pd.DataFrame(columns=columns)
    frame = frame.sort_values("timestamp").copy()
    frame["start"] = frame["timestamp"] - (pd.Timedelta(hours=4) if timestamp_kind == "close" else pd.Timedelta(0))
    frame["valid_bar"] = frame[OHLC].notna().all(axis=1)
    frame["date"] = frame["start"].dt.normalize()
    group = frame.groupby("date")
    result = group.agg(json_bars=("timestamp", "size"), valid_bars=("valid_bar", "sum"),
                       json_open=("open", "first"), json_high=("high", "max"),
                       json_low=("low", "min"), json_close=("close", "last"))
    result["json_complete"] = result["json_bars"].eq(6) & result["valid_bars"].eq(6)
    result.loc[~result["json_complete"], ["json_open", "json_high", "json_low", "json_close"]] = np.nan
    result["json_fear_greed"] = group["Fear & Greed Index"].last() if "Fear & Greed Index" in frame else np.nan
    result["json_fg_distinct"] = group["Fear & Greed Index"].nunique() if "Fear & Greed Index" in frame else 0
    result["coin"] = "BTC"
    return result.reset_index()[columns]


def aggregate_news(documents):
    frame = documents.loc[documents["source_kind"].eq("news")]
    if frame.empty:
        return pd.DataFrame(columns=["date", "news_count", "news_security_count", "news_regulation_count", "news_adoption_count", "news_sources"])
    return frame.groupby("date").agg(news_count=("doc_id", "nunique"), news_security_count=("is_security", "sum"),
                                     news_regulation_count=("is_regulation", "sum"), news_adoption_count=("is_adoption", "sum"),
                                     news_sources=("source", "nunique")).reset_index()


def merge_daily(cleaned, json_daily, news_daily):
    market = cleaned.get("market", pd.DataFrame())
    if market.empty:
        return pd.DataFrame({"date": pd.Series(dtype="datetime64[ns, UTC]"), "coin": pd.Series(dtype="string")})
    result = market.copy()
    if not json_daily.empty:
        result = result.merge(json_daily, on=["date", "coin"], how="left", validate="one_to_one")
    if not news_daily.empty:
        result = result.merge(news_daily, on="date", how="left", validate="many_to_one")
    result["json_matched"] = result["json_bars"].notna() if "json_bars" in result else False
    result["news_matched"] = result["news_count"].notna() if "news_count" in result else False
    if len(result) != len(market) or result.duplicated(["date", "coin"]).any():
        raise ValueError("Слияние нарушило число строк или уникальность date + coin")
    return result.sort_values(["date", "coin"]).reset_index(drop=True)


def upsert_sources(old, incoming):
    """Полная новая строка заменяет старую с тем же ключом, включая её пропуски."""
    result = {}
    for name in FILES:
        previous, batch = old.get(name, pd.DataFrame()), incoming.get(name, pd.DataFrame())
        if batch.empty:
            result[name] = previous.copy()
        elif previous.empty:
            result[name] = batch.copy()
        else:
            result[name] = pd.concat([previous, batch], ignore_index=True).drop_duplicates(KEYS[name], keep="last")
        if not result[name].empty:
            result[name] = result[name].sort_values(KEYS[name]).reset_index(drop=True)
    return result


def prepare_frames(raw, timestamp_kind="open", previous=None):
    incoming, rejected, audit = clean_sources(raw, timestamp_kind)
    cleaned = upsert_sources(previous or {}, incoming)
    documents = annotate_texts(cleaned)
    matrix, vocabulary, idf = vectorize_texts(documents)
    json_daily = aggregate_intraday(cleaned["intraday"], timestamp_kind)
    news_daily = aggregate_news(documents)
    merged = merge_daily(cleaned, json_daily, news_daily)
    profiles = pd.concat([profile_sources(raw, "incoming"), profile_sources(cleaned, "clean_total")], ignore_index=True)
    return {"cleaned": cleaned, "rejected": rejected, "audit": audit, "profiles": profiles,
            "documents": documents, "tfidf": matrix, "vocabulary": vocabulary, "idf": idf,
            "json_daily": json_daily, "news_daily": news_daily, "market_daily": merged,
            "timestamp_kind": timestamp_kind}


In [17]:
TIMESTAMP_KIND = "open"  # подтвердить по описанию JSON: open или close
prepared = prepare_frames(raw_23, timestamp_kind=TIMESTAMP_KIND)
print("Подключённые источники:", list(raw_23) or "нет исходных файлов")
display(prepared["audit"])
if not prepared["profiles"].empty:
    summary = prepared["profiles"][["stage", "dataset", "rows", "columns", "memory_mb"]].drop_duplicates()
    display(summary.round(3))
    for name in raw_23:
        print(f"\n{name}: заполненность до и после очистки")
        display(prepared["profiles"].loc[prepared["profiles"]["dataset"].eq(name),
                ["stage", "attribute", "dtype", "missing", "missing_pct"]].round(2))
        print("Несколько очищенных записей:")
        display(prepared["cleaned"][name].head(3))
print("Записей для ручной проверки:", len(prepared["rejected"]))
if not prepared["rejected"].empty:
    display(prepared["rejected"].head(5))


Подключённые источники: ['market', 'intraday', 'news', 'bots']


,dataset,rule,count
0,market,Не распознана дата date,0
1,market,Не распознано число open,0
2,market,Не распознано число high,0
3,market,Не распознано число low,0
4,market,Не распознано число close,0
...,...,...,...
67,news,Полные повторы: удалены повторные строки,0
68,news,Различающиеся записи с одним ключом,0
69,bots,"Нет ключа, времени или текста",0
70,bots,Полные повторы: удалены повторные строки,0


,stage,dataset,rows,columns,memory_mb
0,incoming,market,30120,32,12.759
32,incoming,intraday,14280,7,2.240
39,incoming,news,413,6,0.385
41,incoming,news,413,6,0.385
45,incoming,bots,220,15,0.347
47,incoming,bots,220,15,0.347
60,clean_total,market,30120,32,11.401
92,clean_total,intraday,14280,7,1.423
99,clean_total,news,413,7,0.401
106,clean_total,bots,220,15,0.355



market: заполненность до и после очистки


,stage,attribute,dtype,missing,missing_pct
0,incoming,date,object,0,0.0
1,incoming,coin,object,0,0.0
2,incoming,open,float64,0,0.0
3,incoming,high,float64,0,0.0
4,incoming,low,float64,0,0.0
...,...,...,...,...,...
87,clean_total,is_halving_day,float64,0,0.0
88,clean_total,cycle_progress_pct,float64,0,0.0
89,clean_total,block_reward_btc,float64,0,0.0
90,clean_total,dxy_close,float64,0,0.0


...


bots: заполненность до и после очистки


,stage,attribute,dtype,missing,missing_pct
45,incoming,record_type,object,0,0.00
46,incoming,question,object,0,0.00
47,incoming,answer,object,0,0.00
48,incoming,bot_category,object,0,0.00
49,incoming,chain,object,0,0.00
50,incoming,section,object,0,0.00
51,incoming,model,object,0,0.00
52,incoming,provider,object,0,0.00
53,incoming,source_url,object,0,0.00
54,incoming,bot_url,object,0,0.00


Несколько очищенных записей:


,record_type,question,answer,bot_category,chain,section,model,provider,source_url,bot_url,tutorial_url,reliability,consensus,critical_note,record_id
0,chain_profile,Which bots and DEXs support Arc?,"Best bots: Maestro, Sigma Bot, Banana Gun, Blo...",chain-selection,Arc,chain_profile,<NA>,<NA>,https://telegramtrading.net/best-telegram-snip...,<NA>,https://telegramtrading.net/best-telegram-snip...,VERIFIED,False,False,chain_arc
1,chain_profile,Which bots and DEXs support Base?,"Best bots: Maestro, Sigma, MevX, Bloom, Banana...",chain-selection,Base,chain_profile,<NA>,<NA>,https://telegramtrading.net/best-base-sniper-bot/,<NA>,https://telegramtrading.net/best-base-sniper-bot/,VERIFIED,False,False,chain_base
2,chain_profile,Which bots and DEXs support BNB Chain (BSC)?,"Best bots: Maestro, MevX, Sigma, Banana Gun, P...",chain-selection,BNB Chain (BSC),chain_profile,<NA>,<NA>,https://telegramtrading.net/best-sniper-bot-fo...,<NA>,https://telegramtrading.net/best-sniper-bot-fo...,VERIFIED,False,False,chain_bsc


Записей для ручной проверки: 0


In [18]:
documents = prepared["documents"]
if not documents.empty:
    print("Примеры аннотирования:")
    display(documents[["doc_id", "source_kind", "text", "topic", "coin_mentions"]].head(5))
    annotation_review = documents.sample(min(20, len(documents)), random_state=42)[["doc_id", "text", "topic", "coin_mentions"]].copy()
    annotation_review["ручная_тема"] = ""
    print("Выборка для ручной проверки (автоматическая разметка ещё не проверена):")
    display(annotation_review.head(5))
    matrix = prepared["tfidf"]
    print("Размер TF-IDF:", matrix.shape, "| Ненулевых весов:", matrix.nnz)
    if matrix.shape[1]:
        # В плотную таблицу переводятся только 3 документа и 8 признаков для просмотра.
        display(pd.DataFrame(matrix[:3, :8].toarray(),
                             index=documents["doc_id"].head(3), columns=prepared["vocabulary"][:8]))
else:
    print("Для аннотирования и векторизации нужны текстовые записи новостей или XML.")

merged_daily = prepared["market_daily"]
if not merged_daily.empty:
    print("Объединённая таблица:", merged_daily.shape)
    selected = [c for c in ["date", "coin", "close", "json_close", "json_complete", "news_count",
                            "news_security_count", "json_matched", "news_matched"] if c in merged_daily]
    display(merged_daily[selected].head(8))
    print("Покрытие источниками, % строк по монетам:")
    display((merged_daily.groupby("coin")[["json_matched", "news_matched"]].mean() * 100).round(2))
    print("Строк после слияния:", len(merged_daily), "| Очищенных строк рынка:", len(prepared["cleaned"]["market"]))
else:
    print("Объединённая таблица не рассчитана: нужен основной рыночный CSV.")


Примеры аннотирования:


,doc_id,source_kind,text,topic,coin_mentions
0,bots:chain_arc,bots,Which bots and DEXs support Arc? Best bots: Ma...,other,
1,bots:chain_base,bots,Which bots and DEXs support Base? Best bots: M...,other,
2,bots:chain_bsc,bots,Which bots and DEXs support BNB Chain (BSC)? B...,other,
3,bots:chain_ethereum,bots,Which bots and DEXs support Ethereum? Best bot...,other,ETH
4,bots:chain_hyperliquid,bots,Which bots and DEXs support Hyperliquid (Hyper...,other,


Выборка для ручной проверки (автоматическая разметка ещё не проверена):


,doc_id,text,topic,coin_mentions,ручная_тема
329,news:41df1d75de371aadce96083ee49affc702693537c...,"Bitcoin slips to $64,000 as oil rally and ETF ...",other,BTC,
247,news:127b9ab5776a43baeb29a71679269e9f38d44bf6d...,Blockchain State Expiry: Why Networks Cannot S...,other,ETH,
390,news:660544125155e2aa4e00a4c79d6929e1b8dab9f7e...,Tether Finally Completes Independent Audit of ...,other,BTC,
145,bots:profile_redotrade,"What is RedoTrade, which chains does it suppor...",security,,
497,news:a639d1a3333b510cd96ae9e03260b0798ea668048...,Bitmine Immersion Technologies Announces Recor...,other,,


Размер TF-IDF: (633, 2000) | Ненулевых весов: 15486


...

Объединённая таблица: (30120, 47)


,date,coin,close,json_close,json_complete,news_count,news_security_count,json_matched,news_matched
0,2017-01-01 00:00:00+00:00,BTC,998.325012,NaN,NaN,NaN,NaN,False,False
1,2017-01-01 00:00:00+00:00,LTC,4.511590,NaN,NaN,NaN,NaN,False,False
2,2017-01-02 00:00:00+00:00,BTC,1021.750000,NaN,NaN,NaN,NaN,False,False
3,2017-01-02 00:00:00+00:00,LTC,4.648940,NaN,NaN,NaN,NaN,False,False
4,2017-01-03 00:00:00+00:00,BTC,1043.839966,NaN,NaN,NaN,NaN,False,False
5,2017-01-03 00:00:00+00:00,LTC,4.626200,NaN,NaN,NaN,NaN,False,False
6,2017-01-04 00:00:00+00:00,BTC,1154.729980,NaN,NaN,NaN,NaN,False,False
7,2017-01-04 00:00:00+00:00,LTC,4.837850,NaN,NaN,NaN,NaN,False,False


Покрытие источниками, % строк по монетам:


,json_matched,news_matched
coin,,
ADA,0.00,1.02
AVAX,0.00,1.50
BNB,0.00,1.02
BTC,66.84,0.93
DOGE,0.00,1.02
DOT,0.00,1.48
ETH,0.00,1.02
LTC,0.00,0.93
SOL,0.00,1.40


Строк после слияния: 30120 | Очищенных строк рынка: 30120


## 2.3.5. Обоснование структуры хранения

Выбирается **набор связанных таблиц в Parquet**: объединённая таблица рынка,
дневные агрегаты, справочник XML и текстовые документы. Для больших текстовых
векторов отдельно используется разреженный NPZ, а не тысячи столбцов с нулями.

| Вариант | Оценка для задачи |
|---|---|
| Один CSV | Прост, но не сохраняет строгие типы, занимает больше места и неудобен для текстовых векторов |
| JSON/XML как единое хранилище | Полезны как исходные форматы, но неудобны для повторного чтения отдельных аналитических столбцов |
| SQLite | Подходит для учебных объёмов и SQL-обновлений; не выбран как основной формат аналитического файлового набора |
| **Parquet + NPZ** | Сохраняет типы и сжатие, позволяет читать нужные столбцы и разделять факты по году; матрица остаётся разреженной |

Фактический объём пока неизвестен. Parquet выбран как масштабируемый формат,
а не как доказательство того, что текущие файлы уже являются «большими данными».
Учебный скрипт pandas обрабатывает данные в памяти и при обновлении пересобирает
агрегаты и TF-IDF; при превышении RAM нужна обработка по разделам, например Arrow/DuckDB/Spark.
Распределённая система для текущего этапа не требуется.

| Файл / каталог внутри версии | Содержимое и связь |
|---|---|
| `sources/market.parquet`, `sources/intraday.parquet`, `sources/news.parquet`, `sources/bots.parquet` | Очищенные источники с ключами для последующего обновления |
| `market_daily/year=...` | Объединённые факты, ключ `coin + date`, разделение по году |
| `json_daily.parquet`, `news_daily.parquet` | Проверяемые дневные агрегаты |
| `documents.parquet` | Тексты и аннотации; `source_record_id` связывает текст с новостью или XML |
| `text_tfidf.npz`, `tfidf_vocabulary.json`, `tfidf_idf.npy` | Матрица, термины по номерам столбцов и веса IDF; строки указаны в `documents.tfidf_row` |
| `profiles.csv`, `audit.csv`, `rejected.csv`, `attribute_dictionary.csv` | Пропуски, протокол очистки, отложенные записи и словарь |
| `manifest.json` | Параметры подготовки, размеры и метаданные версии |

Каждый успешный запуск сохраняет новую папку в `versions`. `CURRENT.json` указывает
на актуальную версию и меняется после успешной записи всех файлов. Предыдущая
версия остаётся доступной. Исходные CSV/XML/JSON не перезаписываются.


In [19]:
def active_version(output_dir):
    root = Path(output_dir)
    pointer = json.loads((root / "CURRENT.json").read_text(encoding="utf-8"))
    version = pointer["version"]
    if not re.fullmatch(r"[0-9a-f]{32}", version):
        raise ValueError("Некорректное имя версии хранилища")
    return root / "versions" / version


def read_previous(output_dir):
    version = active_version(output_dir)
    manifest = json.loads((version / "manifest.json").read_text(encoding="utf-8"))
    frames = {name: pd.read_parquet(version / "sources" / f"{name}.parquet", engine="pyarrow")
              for name in manifest["stored_sources"]}
    return frames, manifest


def write_outputs(bundle, output_dir, dictionary=None):
    """Запись новой версии; CURRENT.json меняется только после успешной записи всех файлов."""
    import pyarrow  # проверяем зависимость до изменения хранилища
    root = Path(output_dir)
    versions = root / "versions"
    versions.mkdir(parents=True, exist_ok=True)
    version_id = uuid.uuid4().hex
    with tempfile.TemporaryDirectory(prefix=".pending_", dir=root) as temporary:
        stage = Path(temporary) / "data"
        stage.mkdir()
        (stage / "sources").mkdir()
        stored = []
        for name, frame in bundle["cleaned"].items():
            if len(frame.columns):
                frame.to_parquet(stage / "sources" / f"{name}.parquet", engine="pyarrow", index=False, compression="snappy")
                stored.append(name)
        fact = bundle["market_daily"]
        if not fact.empty:
            fact.assign(year=fact["date"].dt.year).to_parquet(stage / "market_daily", engine="pyarrow",
                                                            partition_cols=["year"], index=False, compression="snappy")
        else:
            (stage / "market_daily").mkdir()
            fact.to_parquet(stage / "market_daily" / "empty.parquet", engine="pyarrow", index=False)
        for name in ["documents", "json_daily", "news_daily"]:
            bundle[name].to_parquet(stage / f"{name}.parquet", engine="pyarrow", index=False, compression="snappy")
        for name in ["audit", "profiles", "rejected"]:
            bundle[name].to_csv(stage / f"{name}.csv", index=False, encoding="utf-8-sig")
        if dictionary is not None:
            dictionary.to_csv(stage / "attribute_dictionary.csv", index=False, encoding="utf-8-sig")
        sparse.save_npz(stage / "text_tfidf.npz", bundle["tfidf"])
        (stage / "tfidf_vocabulary.json").write_text(json.dumps(bundle["vocabulary"], ensure_ascii=False), encoding="utf-8")
        np.save(stage / "tfidf_idf.npy", bundle["idf"])
        manifest = {"created_at_utc": datetime.now(timezone.utc).isoformat(), "stored_sources": stored,
                    "timestamp_kind": bundle["timestamp_kind"], "fact_rows": len(fact),
                    "document_rows": len(bundle["documents"]), "tfidf_shape": list(bundle["tfidf"].shape),
                    "keys": KEYS, "annotation_method": "keywords_v1", "script_format_version": 1}
        (stage / "manifest.json").write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")
        stage.replace(versions / version_id)
    pointer = root / f".CURRENT_{version_id}.json"
    pointer.write_text(json.dumps({"version": version_id}), encoding="utf-8")
    pointer.replace(root / "CURRENT.json")
    return versions / version_id


In [20]:
SAVE_PREPARED_DATA = False  # True после подключения и проверки исходных файлов
OUTPUT_DIR = Path("prepared_data")
if SAVE_PREPARED_DATA and raw_23:
    catalog_frames = {**raw_23, **prepared["cleaned"], "market_daily": merged_daily, "documents": documents}
    attribute_dictionary = expand_dictionary(dictionary_base, catalog_frames)
    saved_version = write_outputs(prepared, OUTPUT_DIR, attribute_dictionary)
    print("Подготовленные данные сохранены:", saved_version)
else:
    print("Сохранение выключено или исходные файлы отсутствуют. Хранилище не изменено.")


Сохранение выключено или исходные файлы отсутствуют. Хранилище не изменено.


## 2.3.6. Скрипты слияния и обновления

В архиве находится `prepare_data.py` с двумя режимами. Установить зависимости:

```bash
python -m pip install -r requirements.txt
```

Первичное слияние (четыре файла лежат в `data`):

```bash
python prepare_data.py --mode build --data-dir data --out-dir prepared_data
```

Обновление (новые или исправленные **полные записи** лежат в `new_data` под теми же именами):

```bash
python prepare_data.py --mode update --data-dir new_data --out-dir prepared_data
```

При обновлении читаются очищенные источники текущей версии. Новые ключи добавляются,
старые заменяются новыми полными строками, включая их пропуски. Пропущенный файл
не удаляет источник. Строки, не прошедшие проверку, не заменяют сохранённые данные;
причина попадёт в отчёт. Повторная загрузка того же пакета не увеличивает число строк.
Отдельные удаления записей и конкурентные запуски скриптом не реализованы.
Готовые индикаторы CSV не пересчитываются: при исправлении цен источник должен
передать согласованные производные показатели в полной записи.

Агрегаты пересчитываются **после** обновления подробных источников. Поэтому, например,
новые 4h-свечи дополняют неполные сутки. TF-IDF пересчитывается для всего актуального
корпуса: матрица, словарь и IDF сохраняются одной версией, старые номера столбцов
нельзя смешивать с новыми. Правила аннотирования должны иметь одинаковый смысл.

Пример чтения результата после успешного сохранения:

```python
version = active_version("prepared_data")
market = pd.read_parquet(version / "market_daily", columns=["date", "coin", "close"])
texts = pd.read_parquet(version / "documents.parquet")
vectors = sparse.load_npz(version / "text_tfidf.npz")
```
